# Run a synthetic backtest in your browser / 浏览器里的离线回测

No local Python setup or API key is needed in **Google Colab**. Use a standard CPU runtime, review the cells, then choose **Runtime → Run all**. Google sign-in and a runtime may be required; availability is subject to Colab limits. This notebook does not mount Google Drive, request credentials, or call an LLM.

在 Colab 中无需本地安装 Python 或填写 API Key。选择普通 CPU 运行时，检查代码后点击 **代码执行程序 → 全部运行**。可能需要 Google 登录；运行时不保证随时可用。Notebook 不挂载网盘，不读取密钥，不调用大模型。

**Synthetic prices only. This is not historical performance, a strategy recommendation, or investment advice.** All three symbols are fictional. The example teaches the research pipeline, outputs, and portfolio-weight limits. Weekdays are not an exchange calendar, and costs and trading constraints are simplified.

**所有价格均为合成数据，不能当作真实业绩或投资建议。** 三个标的都是虚构的；这里只演示研究流程、输出与组合权重限制。

## 1. Install a fixed project revision / 安装固定版本

The first cell downloads this public project and its dependencies. Installation needs internet access; the remaining cells only use synthetic data and local files. Package dependency versions are resolved at installation time, so the environment is not fully locked.

For local Jupyter, use a dedicated virtual environment. The install cell changes the active notebook environment. If you already imported `qf` from another revision, restart the kernel before running all cells. No GPU or paid runtime is needed for the example.


In [ ]:
import importlib.metadata
import json
import subprocess
import sys

SOURCE_REVISION = "25964e68c1bcbb7b0a231867b52c83c3268a0104"
subprocess.run(
    [sys.executable, "-m", "pip", "install", "--quiet",
     "stock-daily-analyzer @ git+https://github.com/oliverran/stock-daily-analyzer.git@" + SOURCE_REVISION],
    check=True,
)
installed_source = json.loads(importlib.metadata.distribution("stock-daily-analyzer").read_text("direct_url.json"))
assert installed_source["vcs_info"]["commit_id"] == SOURCE_REVISION
print("Installed project revision:", SOURCE_REVISION)


## 2. Run the real demo engine / 运行实际回测引擎

This calls the same `run_demo` used by `stock-analyzer demo`, not a simulated screenshot. Each run creates its own folder and leaves earlier results intact. Expect 60 weekday observations for `DEMO_A`, `DEMO_B`, and `DEMO_C`.

这里调用 CLI 使用的同一个 `run_demo`，不是展示预填结果。每次运行创建新目录，保留此前结果。三个虚构标的各有 60 个工作日的数据。

The inline SVG below is a static overview. Date-to-holdings navigation is available in the exported **report.html**, not in this SVG. Complete section 5 to download and open it.

下方内嵌 SVG 是静态概览，不提供日期与持仓联动。完成第 5 节下载并打开 **report.html** 后，点击图表圆点或日期即可查看当天模拟净值与持仓。


In [ ]:
import json
import tempfile
from pathlib import Path

import pandas as pd
from IPython.display import SVG, display
from qf.demo import run_demo

output_dir = Path(tempfile.mkdtemp(prefix="stock-demo-", dir=Path.cwd()))
summary_path = run_demo(output_dir)
summary = json.loads(summary_path.read_text(encoding="utf-8"))
equity = pd.read_csv(output_dir / "equity.csv", parse_dates=["Date"]).set_index("Date")
holdings = pd.read_csv(output_dir / "holdings.csv", parse_dates=["Date"]).set_index("Date")
assert summary["demo"] is True
assert summary["data_source"] == "deterministic_synthetic"
assert len(equity) == len(holdings) == summary["observations"] == 60
print(summary["warning"])
print("Output folder:", output_dir)
display(SVG(filename=str(output_dir / "equity.svg")))


## 3. Inspect results and assumptions / 查看结果与假设

The portfolio stays in cash until enough factor history exists. `__CASH__` is the cash weight. Daily weights, including cash, sum to 1. The metrics below describe fictional inputs only. Annualizing a short synthetic run does not make it a forecast.

因子历史不足时，组合持有现金。`__CASH__` 表示现金比例，每日权重总和为 1。下面的指标仅描述虚构数据；短期合成结果的年化值不是预测。


In [ ]:
display(pd.Series(summary["metrics"], name="Synthetic demo metrics").to_frame())
display(equity.tail())
display(holdings.tail())
assert (holdings.sum(axis=1) - 1.0).abs().max() < 1e-9
print(json.dumps(summary["config"], indent=2))


## 4. Change one setting / 改一个参数再试试

Edit `TOP_K_VALUES` and rerun this cell. It compares how many fictional symbols the portfolio may hold while keeping the **50% per-symbol cap** and all other settings fixed. In this unconstrained synthetic scenario, TopK=1 leaves the unused weight in cash. Actual trading restrictions can preserve old positions; this is not a guarantee for live data. TopK=2 reproduces the baseline above.

修改 `TOP_K_VALUES` 后重新运行此单元。每个标的仍最多占 50%，其余参数保持不变。在这个没有阻碍换仓的合成场景中，TopK=1 的剩余资金保留为现金。实际交易限制可能保留旧仓，不能据此保证真实数据场景的现金比例。TopK=2 会复现上面的默认结果。

This is a sensitivity demonstration, **not a parameter search for a profitable strategy**. The apparent winner on three manufactured price series has no investment meaning. Outputs from this section are saved separately from the baseline report.


In [ ]:
from dataclasses import asdict

from qf.backtest import DailyBacktester
from qf.demo import SyntheticPriceProvider
from qf.trading import TradingRules
from qf.universe import Stock

TOP_K_VALUES = [1, 2, 3]
if not TOP_K_VALUES or any(type(k) is not int or k not in (1, 2, 3) for k in TOP_K_VALUES):
    raise ValueError("Choose one or more TopK values from 1, 2, 3.")

settings = dict(summary["config"])
trading_rules = TradingRules(**settings.pop("trading"))
run_settings = {key: settings.pop(key) for key in ("start", "end", "rebalance", "initial_cash")}
universe = [Stock(code, code) for code in summary["symbols"]]
rows = []
scenario_holdings = []
for top_k in dict.fromkeys(TOP_K_VALUES):
    engine = DailyBacktester(
        price_provider=SyntheticPriceProvider(),
        trading_rules=trading_rules,
        **{**settings, "top_k": top_k},
    )
    result = engine.run(universe, **run_settings)
    assert len(result.equity) == 60
    assert (result.holdings.sum(axis=1) - 1.0).abs().max() < 1e-9
    if top_k == summary["config"]["top_k"]:
        pd.testing.assert_series_equal(result.equity, equity["equity"], check_names=False, check_freq=False)
    rows.append({"top_k": top_k, "final_cash_weight": result.holdings["__CASH__"].iloc[-1],
                 **asdict(result.metrics)})
    scenario_holdings.append(result.holdings.assign(top_k=top_k).reset_index())
comparison = pd.DataFrame(rows).set_index("top_k")
comparison.to_csv(output_dir / "synthetic-comparison.csv")
pd.concat(scenario_holdings, ignore_index=True).to_csv(output_dir / "synthetic-comparison-holdings.csv", index=False)
print("Synthetic inputs only. These are not real strategy results.")
display(comparison)


## 5. Keep your results / 保存结果

This cell creates a ZIP with the five baseline files plus the two comparison CSVs. It contains only the named demo files, not other notebook files or credentials. In Colab, open the **Files** sidebar, refresh it, find the printed ZIP name, and use its menu to download. Extract the whole archive before opening `report.html` so its data links work.

此单元将默认演示的 5 个文件与 2 个参数对比 CSV 打包。仅包含列出的演示文件。在 Colab 左侧 **文件** 面板刷新，找到打印出的 ZIP 文件，通过菜单下载。完整解压后打开 `report.html`，其中链接的数据文件才能正常访问。报告属于默认 TopK=2；参数对比在独立 CSV 中。

Download results before the runtime expires. Files in a temporary runtime do not persist automatically.

### Open the interactive report / 打开可交互报告

1. Download the ZIP from the Colab Files sidebar and extract **all** files into one folder. Do not open the HTML inside the ZIP preview.
2. Open `report.html` in a local browser. Select a chart point or a date-table link; keyboard users can Tab to a date and press Enter. Browser Back/Forward restores selections.
3. The selected holdings are the simulation’s post-processing records, not trade signals, execution reasons, or return attribution. The report uses native links and CSS, with no JavaScript or external requests.

1. 从 Colab 文件面板下载 ZIP，将**全部**文件解压到同一个目录，不要在压缩包预览内打开 HTML。
2. 用本地浏览器打开 `report.html`，点击图表圆点或日期表链接；也可用 Tab 选中日期后按 Enter，浏览器后退 / 前进可恢复选择。
3. 持仓仅为模拟处理后的记录，不解释交易信号、成交原因或收益归因。报告使用原生链接与 CSS，不执行 JavaScript，也不请求外部服务。

Colab output frames and GitHub notebook previews are not the full report viewer; the downloaded HTML is the supported way to use these controls.


In [ ]:
from zipfile import ZIP_DEFLATED, ZipFile

export_names = ["equity.csv", "holdings.csv", "summary.json", "equity.svg", "report.html",
                "synthetic-comparison.csv", "synthetic-comparison-holdings.csv"]
archive_path = output_dir.with_suffix(".zip")
with ZipFile(archive_path, "w", compression=ZIP_DEFLATED) as archive:
    for name in export_names:
        archive.write(output_dir / name, arcname=name)
print("Download:", archive_path.name)


## Next steps / 下一步

- [Project and local installation / 项目与本地安装](https://github.com/oliverran/stock-daily-analyzer)
- [English guide](https://github.com/oliverran/stock-daily-analyzer/blob/main/README.en.md)
- [Report a reproducible problem / 反馈可复现的问题](https://github.com/oliverran/stock-daily-analyzer/issues): include the command or cell, Python version, and sanitized error; never include API keys.

If this helps you understand the pipeline, a Star helps you find it again. Live A-share data, daily reports, and optional LLM analysis are separate workflows requiring their own checks. This notebook validates none of those services.
